In [1]:
import os
import gc
import glob
from collections import defaultdict

from pprint import pprint
import requests

import torch
import pickle
import numpy as np
import pandas as pd

In [2]:
def process_obs_and_action_dims(i, data, dims_df):
    tasks_to_ids = {task_id: -1 for task_id in data['task'].unique().tolist()}
    
    for task_id in tasks_to_ids.keys():
        tasks_to_ids[task_id] = (data['task']==task_id).nonzero(as_tuple=True)[0][0].item()
    
    for task_id, traj_idx in tasks_to_ids.items():
        obs_dim = data['obs'][traj_idx][0].count_nonzero().item()
        action_dim = data['action'][traj_idx][0].count_nonzero().item()

        dims_df.loc[-1] = [task_id, obs_dim, action_dim]
    dims_df.to_csv(f'./dims_df_{i}.csv')

In [3]:
for i in range(13,20):
    print(f'Processing chunk {i}...')

    dims_df = pd.DataFrame({
        'Task ID': pd.Series(dtype='int'),
        'Observation Dim': pd.Series(dtype='int'),
        'Action Dim': pd.Series(dtype='int')
    })
    
    with torch.serialization.safe_globals(["tensordict.tensordict.TensorDict"]):
        data = torch.load(f'./chunk_{i}.pt', weights_only=False)
    
    process_obs_and_action_dims(i, data, dims_df)
    
    del data
    del dims_df
    gc.collect()

Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...


In [2]:
dims_df = pd.concat(map(pd.read_csv, glob.glob(os.path.join('./', "dims_df_*.csv"))), ignore_index=True)

dims_df.head()

,Unnamed: 0,Task ID,Observation Dim,Action Dim
0,-1,11,5,1
1,-1,19,15,4
2,-1,7,2,1
3,-1,28,2,2
4,-1,1,15,5


In [3]:
CART_POLE_OBS_DIM = 5
CART_POLE_ACTION_DIM = 1
CART_POLE_NUM_TASKS = 4

CHEETAH_OBS_DIM = 17
CHEETAH_ACTION_DIM = 6
CHEETAH_NUM_TASKS = 5

CUP_OBS_DIM = 8
CUP_ACTION_DIM = 2
CUP_NUM_TASKS = 2

HOPPER_OBS_DIM = 15
HOPPER_ACTION_DIM = 4
HOPPER_NUM_TASKS = 3

WALKER_OBS_DIM = 24
WALKER_ACTION_DIM = 6
WALKER_NUM_TASKS = 5

In [4]:
print('Cart Pole:')
dims_df[dims_df['Observation Dim']==CART_POLE_OBS_DIM].groupby('Task ID').first().head(n=100)

Cart Pole:


,Unnamed: 0,Observation Dim,Action Dim
Task ID,,,
9,-1,5,1
11,-1,5,1
18,-1,5,4
25,-1,5,4
27,-1,5,3


In [5]:
print('Cheetah:')
dims_df[dims_df['Observation Dim']==CHEETAH_OBS_DIM].groupby('Task ID').first().head(n=100)

Cheetah:


,Unnamed: 0,Observation Dim,Action Dim
Task ID,,,
22,-1,17,5
23,-1,17,3
24,-1,17,6


In [6]:
print('Cup:')
dims_df[dims_df['Observation Dim']==CUP_OBS_DIM].groupby('Task ID').first().head(n=100)

Cup:


,Unnamed: 0,Observation Dim,Action Dim
Task ID,,,


In [7]:
print('Hopper:')
dims_df[dims_df['Observation Dim']==HOPPER_OBS_DIM].groupby('Task ID').first().head(n=100)

Hopper:


,Unnamed: 0,Observation Dim,Action Dim
Task ID,,,
1,-1,15,5
2,-1,15,5
19,-1,15,4


In [8]:
print('Walker:')
dims_df[dims_df['Observation Dim']==WALKER_OBS_DIM].groupby('Task ID').first().head(n=100)

Walker:


,Unnamed: 0,Observation Dim,Action Dim
Task ID,,,
